In [ ]:
%pip -q install -U youtube-transcript-api

from youtube_transcript_api import YouTubeTranscriptApi
from urllib.parse import urlparse, parse_qs
from pathlib import Path
import re


def extract_video_id(url_or_id: str) -> str:
    value = url_or_id.strip()

    if re.fullmatch(r"[\w-]{11}", value):
        return value

    parsed = urlparse(value)
    host = parsed.netloc.lower()
    path_parts = [part for part in parsed.path.split("/") if part]

    video_id = ""

    if host in {"youtu.be", "www.youtu.be"} and path_parts:
        video_id = path_parts[0]

    elif "youtube.com" in host:
        if parsed.path == "/watch":
            video_id = parse_qs(parsed.query).get("v", [""])[0]

        elif len(path_parts) >= 2 and path_parts[0] in {"shorts", "live", "embed"}:
            video_id = path_parts[1]

    if not re.fullmatch(r"[\w-]{11}", video_id):
        raise ValueError("올바른 유튜브 링크 또는 영상 ID를 입력해 주세요.")

    return video_id


def format_timestamp(seconds: float) -> str:
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"

    return f"{minutes:02d}:{secs:02d}"


# 유튜브 링크 입력
youtube_url = input("유튜브 링크를 붙여넣으세요: ").strip()
video_id = extract_video_id(youtube_url)

# 한국어 우선, 없으면 영어 자막
api = YouTubeTranscriptApi()
transcript = api.fetch(video_id, languages=["ko", "en"])

plain_lines = []
timestamp_lines = []

for snippet in transcript:
    text = snippet.text.replace("\n", " ").strip()

    if not text:
        continue

    plain_lines.append(text)
    timestamp_lines.append(f"[{format_timestamp(snippet.start)}] {text}")

plain_text = "\n".join(plain_lines)
timestamp_text = "\n".join(timestamp_lines)

# 파일 저장
plain_path = Path(f"youtube_transcript_{video_id}_plain.txt").resolve()
timestamp_path = Path(f"youtube_transcript_{video_id}_timestamps.txt").resolve()

plain_path.write_text(plain_text, encoding="utf-8-sig")
timestamp_path.write_text(timestamp_text, encoding="utf-8-sig")

# 기본 정보 출력
print("\n" + "=" * 70)
print("추출 완료")
print("=" * 70)
print(f"영상 ID: {video_id}")
print(f"자막 언어: {transcript.language}")
print(f"자동 생성 자막 여부: {transcript.is_generated}")
print(f"추출된 자막 조각 수: {len(transcript)}")

print("\n저장 위치:")
print(f"일반 텍스트: {plain_path}")
print(f"타임스탬프 포함: {timestamp_path}")

# 자막 전체 출력
print("\n" + "=" * 70)
print("자막 전체 내용")
print("=" * 70 + "\n")
print(timestamp_text)

import os
import subprocess

# 일반 자막 전체를 윈도우 클립보드에 자동 복사
subprocess.run(
    "clip",
    input=plain_text,
    text=True,
    shell=True,
    check=True
)

# 저장된 파일이 있는 폴더를 탐색기로 자동 열기
os.startfile(plain_path.parent)

print("\n일반 자막 전체가 클립보드에 복사되었습니다.")
print("원하는 곳에서 Ctrl + V만 누르시면 됩니다.")